# NeuroFinance AI — Phase 5 & 6: FT-Transformer Model Training & Evaluation
This notebook tracks the training performance of the PyTorch FT-Transformer model, computes evaluation metrics on the test dataset, and analyzes model calibration.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, classification_report, roc_auc_score, auc
from sklearn.calibration import calibration_curve

# Add project root to path for imports
sys.path.append("..")
from models.ft_transformer import FTTransformer
from models.train import TabularDataset

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 1. Load Trained Model and Test Dataset

In [ ]:
test_path = "../data/processed/test.csv"
model_path = "../models/ft_transformer.pt"

test_dataset = TabularDataset(test_path)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False)

num_features = test_dataset.X.shape[1]
print(f"Test Set Size: {len(test_dataset)}")
print(f"Number of Features: {num_features}")

# Load model
model = FTTransformer(num_features=num_features, d_token=32, n_blocks=2, n_heads=4, d_ffn=64, dropout=0.1)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()
print("Model loaded successfully!")

## 2. Generate Test Predictions

In [ ]:
all_probs = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.numpy())
        all_targets.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_targets = np.array(all_targets)
all_preds = (all_probs > 0.5).astype(int)
print("Predictions complete!")

## 3. ROC-AUC and PR-AUC Evaluation

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(all_targets, all_probs)
roc_auc = roc_auc_score(all_targets, all_probs)

# PR Curve
precision, recall, _ = precision_recall_curve(all_targets, all_probs)
pr_auc = auc(recall, precision)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ROC
ax1.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.4f})")
ax1.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Receiver Operating Characteristic (ROC) Curve")
ax1.legend(loc="lower right")

# Plot PR
ax2.plot(recall, precision, color="purple", lw=2, label=f"PR Curve (AUC = {pr_auc:.4f})")
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall (PR) Curve")
ax2.legend(loc="lower left")

plt.show()

## 4. Confusion Matrix and Classification Report

In [ ]:
conf_mat = confusion_matrix(all_targets, all_preds)
print("Classification Report:")
print(classification_report(all_targets, all_preds))

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Repaid (0)", "Default (1)"],
            yticklabels=["Repaid (0)", "Default (1)"])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

## 5. Probability Calibration Curve

In [ ]:
prob_true, prob_pred = calibration_curve(all_targets, all_probs, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker="s", color="green", label="FT-Transformer")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly Calibrated")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title("Probability Calibration Curve")
plt.legend(loc="lower right")
plt.show()